# 01 – Exploratory Data Analysis (EDA)

**Project:** E-commerce Purchase Behavior Prediction and Customer Segmentation Using Machine Learning  
**Dataset:** Retailrocket E-commerce Clickstream Dataset (`data/raw/events.csv`)

Questions answered in this notebook:
1. How big is the raw clickstream dataset and what does each column represent?
2. What data quality issues exist (missing values, exact duplicate rows, timestamp ordering)?
3. What is the distribution of `view`, `addtocart`, and `transaction` events?
4. How is the early-session purchase target (`will_purchase`) distributed and which features correlate with conversion?

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml import config

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Load the Raw Retailrocket Clickstream Data

In [ ]:
raw_events = pd.read_csv(config.RAW_DATA_PATH)
print("Raw events shape:", raw_events.shape)
raw_events.head()

## 2. Data Quality & Event Distribution Checks

In [ ]:
quality = pd.DataFrame({
    "dtype": raw_events.dtypes.astype(str),
    "missing": raw_events.isna().sum(),
    "missing_%": (raw_events.isna().mean() * 100).round(2),
    "unique": raw_events.nunique(),
})
print("Exact duplicate rows:", raw_events.duplicated().sum())
print("Event counts:\n", raw_events["event"].value_counts())
quality

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
raw_events["event"].value_counts().plot.bar(ax=ax, color=["#2F855A", "#D69E2E", "#153E2D"])
ax.set_title("Retailrocket Clickstream Event Distribution")
ax.set_ylabel("Event Count")
plt.tight_layout(); plt.show()

## 3. Explore Processed Early-Session Modeling Data (`clean.csv`) & Target Balance

In [ ]:
df = pd.read_csv(config.PROCESSED_DATA_PATH)
print("Modeling dataset shape:", df.shape)
print("Target balance:\n", df[config.TARGET_COLUMN].value_counts(normalize=True).round(4))
df.head()

In [ ]:
num_cols = ["views_count", "unique_items_viewed", "add_to_cart_count", "elapsed_seconds", "prior_sessions", "prior_purchases"]
corr = df[num_cols].assign(will_purchase=(df[config.TARGET_COLUMN] == "Yes").astype(int)).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Greens")
plt.title("Correlation Matrix (Early-Session Features vs Purchase Conversion)")
plt.tight_layout(); plt.show()

## 4. Key EDA Findings ✍️

- **Raw Data Volume:** `2,756,101` events across `1,407,580` unique visitors and `235,061` products (`2015-05-03` to `2015-09-18`).
- **Data Quality Issues:** `460` exact duplicate rows; timestamps are not pre-sorted; `transactionid` is missing on `99.19%` of rows by design (only present on `transaction` events).
- **Severe Funnel Drop-Off:** `view` (`96.67%`) $\rightarrow$ `addtocart` (`2.52%`) $\rightarrow$ `transaction` (`0.81%`).
- **Target Imbalance:** In the leakage-safe 3-event snapshot dataset (`clean.csv`), `~7.6%` of sessions convert (`will_purchase = Yes`). Because of class imbalance, **F1-score, Recall, Precision, ROC-AUC, and PR-AUC** must be used rather than accuracy alone.
- **Top Conversion Signals:** Early `add_to_cart_count`, `ends_with_cart = Yes`, and historical `prior_purchases` show the strongest positive association with purchase conversion.